# Datenverständnis und Datenaufbereitung

Dieses Notebook dient der Untersuchung und Bereinigung der ursprünglichen Rotten-Tomatoes-Daten. Die bereinigten Rezensionen bilden anschließend die Grundlage für die Klassifikation, während der zusammengeführte Datensatz hauptsächlich für die explorative Datenanalyse verwendet wird.

Ziele dieses Notebooks:

1. Laden und Verstehen der ursprünglichen Datensätze
2. Prüfung zentraler Attribute und möglicher fehlender Werte
3. Bereinigung der Rezensionstexte für die spätere Klassifikation
4. Zusammenführung von Rezensionen und Filminformationen für die explorative Datenanalyse
5. Speicherung der bereinigten Rezensionen für die Klassifikation sowie eines zusammengeführten Datensatzes für die explorative Analyse

## Import

In [2]:
import pandas as pd

# Bibliothek zur Bereinigung von Texten mithilfe regulärer Ausdrücke
import re

import matplotlib.pyplot as plt

In [13]:
#für no English Cleaning
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

## Rohdaten laden

Es werden zwei ursprüngliche Datensätze geladen:

- `rotten_tomatoes_movie_reviews.csv`: einzelne Kritikerrezensionen
- `rotten_tomatoes_movies.csv`: Metadaten zu den Filmen

Die Rezensionen enthalten unter anderem den Rezensionstext und eine Film-ID.
Die Filmdaten enthalten zusätzliche Informationen wie Titel, Genre, Laufzeit und Bewertungen.

In [3]:
movie_reviews = pd.read_csv('../Dataset/rotten_tomatoes_movie_reviews.csv', sep=',', na_values=[''], quotechar='"')
movie_reviews.head()

,id,reviewId,creationDate,criticName,isTopCritic,originalScore,reviewState,publicatioName,reviewText,scoreSentiment,reviewUrl
0,beavers,1145982,2003-05-23,Ivan M. Lincoln,False,3.5/4,fresh,Deseret News (Salt Lake City),Timed to be just long enough for most youngste...,POSITIVE,http://www.deseretnews.com/article/700003233/B...
1,blood_mask,1636744,2007-06-02,The Foywonder,False,1/5,rotten,Dread Central,It doesn't matter if a movie costs 300 million...,NEGATIVE,http://www.dreadcentral.com/index.php?name=Rev...
2,city_hunter_shinjuku_private_eyes,2590987,2019-05-28,Reuben Baron,False,NaN,fresh,CBR,The choreography is so precise and lifelike at...,POSITIVE,https://www.cbr.com/city-hunter-shinjuku-priva...
3,city_hunter_shinjuku_private_eyes,2558908,2019-02-14,Matt Schley,False,2.5/5,rotten,Japan Times,The film's out-of-touch attempts at humor may ...,NEGATIVE,https://www.japantimes.co.jp/culture/2019/02/0...
4,dangerous_men_2015,2504681,2018-08-29,Pat Padua,False,NaN,fresh,DCist,Its clumsy determination is endearing and some...,POSITIVE,http://dcist.com/2015/11/out_of_frame_dangerou...


In [4]:
movies = pd.read_csv('../Dataset/rotten_tomatoes_movies.csv', sep=',', na_values=[''], quotechar='"')
movies.head()

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix
0,space-zombie-bingo,Space Zombie Bingo!,50.0,NaN,NaN,NaN,NaN,2018-08-25,75.0,"Comedy, Horror, Sci-fi",English,George Ormrod,"George Ormrod,John Sabotta",NaN,NaN,NaN
1,the_green_grass,The Green Grass,NaN,NaN,NaN,NaN,NaN,2020-02-11,114.0,Drama,English,Tiffany Edwards,Tiffany Edwards,NaN,NaN,NaN
2,love_lies,"Love, Lies",43.0,NaN,NaN,NaN,NaN,NaN,120.0,Drama,Korean,"Park Heung-Sik,Heung-Sik Park","Ha Young-Joon,Jeon Yun-su,Song Hye-jin",NaN,NaN,NaN
3,the_sore_losers_1997,Sore Losers,60.0,NaN,NaN,NaN,NaN,2020-10-23,90.0,"Action, Mystery & thriller",English,John Michael McCarthy,John Michael McCarthy,NaN,NaN,NaN
4,dinosaur_island_2002,Dinosaur Island,70.0,NaN,NaN,NaN,NaN,2017-03-27,80.0,"Fantasy, Adventure, Animation",English,Will Meugniot,John Loy,NaN,NaN,NaN


## Erste Übersicht über die Datensätze

In diesem Schritt werden Größe, Spaltennamen und IDs der beiden Datensätze untersucht.
Damit wird geprüft, welche Informationen verfügbar sind und ob die Datensätze über die gemeinsame Variable `id` verbunden werden können.

In [5]:
# Anzahl der Zeilen und Spalten im Filmdatensatz
movies.shape

(143258, 16)

In [6]:
# Anzahl der Zeilen und Spalten im Rezensionsdatensatz
movie_reviews.shape

(1444963, 11)

In [7]:
# Spalten des Filmdatensatzes
movies.columns

Index(['id', 'title', 'audienceScore', 'tomatoMeter', 'rating',
       'ratingContents', 'releaseDateTheaters', 'releaseDateStreaming',
       'runtimeMinutes', 'genre', 'originalLanguage', 'director', 'writer',
       'boxOffice', 'distributor', 'soundMix'],
      dtype='str')

In [8]:
# Spalten des Rezensionsdatensatzes
movie_reviews.columns

Index(['id', 'reviewId', 'creationDate', 'criticName', 'isTopCritic',
       'originalScore', 'reviewState', 'publicatioName', 'reviewText',
       'scoreSentiment', 'reviewUrl'],
      dtype='str')

In [9]:
# Beispielhafte Film-IDs im Filmdatensatz
movies["id"].head()

0      space-zombie-bingo
1         the_green_grass
2               love_lies
3    the_sore_losers_1997
4    dinosaur_island_2002
Name: id, dtype: str

In [10]:
# Beispielhafte Film-IDs im Rezensionsdatensatz
movie_reviews["id"].head()

0                              beavers
1                           blood_mask
2    city_hunter_shinjuku_private_eyes
3    city_hunter_shinjuku_private_eyes
4                   dangerous_men_2015
Name: id, dtype: str

In [11]:
# Anzahl eindeutiger Filme im Filmdatensatz
movies["id"].nunique()

142052

In [12]:
# Anzahl eindeutiger Filme im Rezensionsdatensatz
movie_reviews["id"].nunique()

69263

## Erste Zusammenführung der Datensätze

Die beiden Datensätze werden über die gemeinsame Spalte `id` zusammengeführt.
Dies dient zunächst nur dem Datenverständnis und der Prüfung, ob die Zuordnung zwischen Rezensionen und Filmen funktioniert.

Für die spätere Klassifikation wird zunächst nur der Rezensionstext verwendet.
Für Regression und Clustering sind jedoch die Filminformationen aus dem zweiten Datensatz relevant.

In [42]:
merged = movie_reviews.merge (movies, on="id", how="left")
merged.head()

,id,reviewId,creationDate,criticName,isTopCritic,originalScore,reviewState,publicatioName,reviewText,scoreSentiment,...,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix
0,beavers,1145982,2003-05-23,Ivan M. Lincoln,False,3.5/4,fresh,Deseret News (Salt Lake City),Timed to be just long enough for most youngste...,POSITIVE,...,NaN,2011-06-21,30.0,Documentary,English,Stephen Low,NaN,NaN,NaN,NaN
1,blood_mask,1636744,2007-06-02,The Foywonder,False,1/5,rotten,Dread Central,It doesn't matter if a movie costs 300 million...,NEGATIVE,...,NaN,NaN,NaN,NaN,NaN,Unknown Director,NaN,NaN,NaN,NaN
2,city_hunter_shinjuku_private_eyes,2590987,2019-05-28,Reuben Baron,False,NaN,fresh,CBR,The choreography is so precise and lifelike at...,POSITIVE,...,NaN,NaN,NaN,NaN,NaN,Kenji Kodama,NaN,NaN,NaN,NaN
3,city_hunter_shinjuku_private_eyes,2558908,2019-02-14,Matt Schley,False,2.5/5,rotten,Japan Times,The film's out-of-touch attempts at humor may ...,NEGATIVE,...,NaN,NaN,NaN,NaN,NaN,Kenji Kodama,NaN,NaN,NaN,NaN
4,dangerous_men_2015,2504681,2018-08-29,Pat Padua,False,NaN,fresh,DCist,Its clumsy determination is endearing and some...,POSITIVE,...,NaN,NaN,NaN,NaN,NaN,Unknown Director,NaN,NaN,NaN,NaN


## Überblick über den zusammengeführten Datensatz

Nach dem Merge wird geprüft, wie groß der zusammengeführte Datensatz ist und in welchen zentralen Variablen fehlende Werte auftreten.

In [43]:
merged.shape


(1469840, 26)

In [44]:
merged[["reviewText", "title", "audienceScore", "tomatoMeter", "genre"]].isnull().sum()

reviewText       70285
title             5767
audienceScore    70546
tomatoMeter      71570
genre            23747
dtype: int64

### Erste Beobachtungen

- Die Datensätze können über die gemeinsame Variable `id` zusammengeführt werden.
- Der Rezensionsdatensatz enthält eine sehr große Anzahl einzelner Kritikerrezensionen.
- Für die Klassifikation ist vor allem `reviewText` relevant.
- Für spätere Analysen auf Filmebene sind zusätzlich Variablen wie `audienceScore`, `tomatoMeter`, `runtimeMinutes`, `genre` und `originalLanguage` wichtig.
- Fehlende Werte müssen je nach späterer Fragestellung unterschiedlich behandelt werden.

# Datenbereinigung

Im nächsten Schritt werden die Rezensionen bereinigt.
Ziel ist es, einen Datensatz mit englischsprachigen Rezensionen ausreichender Textqualität zu erstellen.

Dieser Datensatz bildet später die Grundlage für:

- automatische Emotionserkennung,
- Klassifikation der emotionalen Intensität,
- Aggregation der Emotionswerte auf Filmebene.

## Bereinigung der Rezensionen

Zunächst wird eine Kopie des ursprünglichen Rezensionsdatensatzes erstellt.
Danach werden fehlende Texte, Duplikate und sehr kurze Rezensionen entfernt.

In [14]:
reviews_clean = movie_reviews.copy()
reviews_clean.shape

(1444963, 11)

In [15]:
# Fehlende Rezensionstexte entfernen
reviews_clean = reviews_clean.dropna(subset=["reviewText"])

# Doppelte Einträge entfernen
reviews_clean = reviews_clean.drop_duplicates()
reviews_clean.shape

(1363875, 11)

In [16]:
# Wortanzahl pro Rezension berechnen
reviews_clean["word_count"] = (
    reviews_clean["reviewText"]
    .str.split()
    .str.len()
)

# Sehr kurze Rezensionen entfernen
# Begründung: Texte mit weniger als fünf Wörtern enthalten meist zu wenig Information
# für eine zuverlässige Emotionserkennung und Klassifikation.
reviews_clean = reviews_clean[
    reviews_clean["word_count"] >= 5
]
reviews_clean.shape

(1332661, 12)

## Sprachfilter

Da das verwendete Emotionserkennungsmodell auf englischsprachige Texte ausgelegt ist, werden nur englische Rezensionen beibehalten.

Nicht-englische Rezensionen könnten zu fehlerhaften Emotionsscores führen und werden daher ausgeschlossen.

In [17]:
def detect_language(text):
    """Erkennt die Sprache eines Rezensionstextes.

    Falls die Spracherkennung fehlschlägt, wird 'unknown' zurückgegeben.
    """
    try:
        return detect(str(text))
    except LangDetectException:
        return "unknown"

# Sprache jeder Rezension erkennen
# Achtung: Dieser Schritt kann bei sehr großen Datensätzen einige Zeit dauern.
reviews_clean["language"] = (
    reviews_clean["reviewText"]
    .apply(detect_language)
)

# Nur englischsprachige Rezensionen behalten
reviews_clean = reviews_clean[
    reviews_clean["language"] == "en"
]

reviews_clean["language"].value_counts()

language
en    1323618
Name: count, dtype: int64

## Zusätzliche Qualitätsprüfung der Rezensionstexte

Einige Texte können sehr viele Sonderzeichen, HTML-Reste oder andere nicht alphabetische Zeichen enthalten.
Um solche Fälle zu erkennen, wird der Anteil alphabetischer Zeichen am gesamten Text berechnet.

Dieser Wert wird als `letter_ratio` bezeichnet.

In [18]:
"""Berechnet den Anteil alphabetischer Zeichen an allen Zeichen eines Textes."""
def letter_ratio(text):
    text = str(text)
    if len(text) == 0:
        return 0
    letters = len(re.findall(r"[A-Za-z]", text))
    return letters / len(text)

reviews_clean["letter_ratio"] = (
    reviews_clean["reviewText"]
    .apply(letter_ratio)
)

reviews_clean["letter_ratio"].describe()


count    1.323618e+06
mean     8.053644e-01
std      3.155136e-02
min      2.461538e-01
25%      7.909091e-01
50%      8.089888e-01
75%      8.250000e-01
max      9.230769e-01
Name: letter_ratio, dtype: float64

In [20]:
# Beispiele mit besonders niedriger Letter-Ratio anzeigen
reviews_clean.sort_values(
    by="letter_ratio"
)[["reviewText", "letter_ratio"]].head(20)

,reviewText,letter_ratio
563041,At the wedding&#44; at the grocery&#44; at the...,0.246154
1097345,&#8220;Inspired by real historical events&#822...,0.323636
1289083,As the kids say&#44; &#8220;it&#8217;s a vibe&...,0.350877
1293684,&#8220;GIVE ME A F &#8211; GIVE ME A U &#8211;...,0.352113
493012,&quot;I&apos;m a real boy&#33;&quot;&#10;&#46;...,0.368932
211523,&#46;&#46;&#46;a pleasant piece of work&#46;&#...,0.370370
314603,&#46;&#46;&#46;an amiable piece of work&#46;&#...,0.370370
445155,&#46;&#46;&#46;a stirring piece of work&#46;&#...,0.370370
315284,&#46;&#46;&#46;a solid debut from Novak&#46;&#...,0.370370
1346722,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...,0.372000


In [25]:
# Anzahl der Rezensionen mit sehr niedriger Letter-Ratio
(reviews_clean["letter_ratio"] < 0.5).sum()

np.int64(85)

In [26]:
# Beispiele für entfernte Rezensionen prüfen
reviews_clean[
    reviews_clean["letter_ratio"] < 0.5
][["reviewText", "letter_ratio"]].head(30)

,reviewText,letter_ratio
4620,For a while it seems &#8216;Dominion&#8217; wa...,0.470000
44126,&#46;&#46;&#46;an absolutely redundant piece o...,0.477612
91796,&#46;&#46;&#46;a slow-moving yet mostly intrig...,0.492958
99380,... the best movie of 1998...,0.482759
116572,... a near miss ...,0.473684
139568,My #1 movie of 2018.,0.450000
145667,&#46;&#46;&#46;the feel-good movie of the year...,0.409836
153326,Here&#8217;s the 411 on The 355 &#8212; it&#82...,0.400000
185879,&#46;&#46;&#46;an erratic yet mostly rewarding...,0.485294
198317,You can&#8217;t spell &#8220;menacing&#8221; w...,0.440476


In [27]:
# Rezensionen mit sehr niedriger Letter-Ratio entfernen
reviews_clean = reviews_clean[
    reviews_clean["letter_ratio"] >= 0.5
]
reviews_clean.shape
# Additional data quality checks identified only 104 reviews (0.008% of the dataset) with a very low proportion of alphabetic characters. These reviews were removed.

(1323533, 14)

### Ergebnis der Review-Bereinigung

Nach der Bereinigung enthält `reviews_clean` nur noch Rezensionen, die:

- einen vorhandenen Rezensionstext besitzen,
- nicht doppelt im Datensatz vorkommen,
- mindestens fünf Wörter enthalten,
- als englischsprachig erkannt wurden,
- eine ausreichende Textqualität auf Basis der `letter_ratio` besitzen.

Dieser Datensatz wird anschließend als `reviews_clean.csv` gespeichert und bildet die Grundlage für die Emotionserkennung und Klassifikation.

## Bereinigung der Filmdaten

Für spätere Analysen auf Filmebene wird auch der Filmdatensatz bereinigt.
Dabei werden fehlende IDs und doppelte Einträge entfernt.

In [30]:
movies_clean = movies.copy()
movies_clean.shape


(143258, 16)

In [31]:
# Filme ohne ID und doppelte Einträge entfernen
movies_clean = movies_clean.dropna(subset=["id"])

movies_clean = movies_clean.drop_duplicates()

movies_clean.shape


(142054, 16)

## Zusammenführen der bereinigten Rezensionen und Filmdaten

Die bereinigten Rezensionen werden mit den bereinigten Filmdaten verbunden.
Der resultierende Datensatz `merged_clean` enthält sowohl den Rezensionstext als auch die zugehörigen Filminformationen.

Dieser Datensatz ist besonders für spätere Analysen auf Filmebene relevant.

In [32]:
merged_clean = reviews_clean.merge (movies_clean, on="id", how="left")
merged_clean.shape

(1323674, 29)

In [33]:
# Rezensionen ohne zugeordneten Filmtitel entfernen
merged_clean = merged_clean.dropna(subset=["title"])
merged_clean.shape

(1318419, 29)

## Prüfung zentraler Bewertungsvariablen

Für die spätere Regression sind insbesondere `audienceScore` und `tomatoMeter` relevant.
Daher wird geprüft, wie viele Bewertungen im bereinigten Datensatz vorhanden sind.

In [34]:
# Anzahl der Rezensionen mit vorhandenem audienceScore
merged_clean["audienceScore"].notna().sum()

np.int64(1258535)

In [35]:
# Anzahl eindeutiger Filme mit vorhandenem audienceScore
merged_clean.dropna(
    subset=["audienceScore"]
)["id"].nunique()

48450

# Bereinigte Datensätze speichern

Am Ende werden zwei bereinigte Dateien gespeichert:

- `reviews_clean.csv`: bereinigte Rezensionen für Emotionserkennung und Klassifikation
- `merged_clean.csv`: bereinigte Rezensionen mit Filminformationen für spätere Analysen

Diese Dateien werden in den folgenden Notebooks weiterverwendet.

In [29]:
reviews_clean.to_csv(
    "../Dataset/reviews_clean.csv",
    index=False
)

In [36]:
merged_clean.to_csv(
    "../Dataset/merged_clean.csv",
    index=False
)

# Zusammenfassung

In diesem Notebook wurden die ursprünglichen Rotten-Tomatoes-Daten geladen, untersucht und bereinigt.

Die wichtigsten Ergebnisse:

- Der Rezensionsdatensatz wurde auf englischsprachige Rezensionen ausreichender Qualität reduziert.
- Sehr kurze Rezensionen, fehlende Texte, Duplikate und Texte mit einer sehr niedrigen `letter_ratio` wurden entfernt.
- Die bereinigten Rezensionen wurden zusätzlich mit den Filmdaten zusammengeführt, um die Datenstruktur und ihre Eigenschaften zu analysieren.
- Die Datei `reviews_clean.csv` bildet die Grundlage für die nachfolgende Klassifikation. Dort werden die Rezensionen um automatisch berechnete Emotionsmerkmale erweitert.
- Der daraus entstehende Datensatz dient anschließend als Ausgangspunkt für die Regressions- und Clustering-Modelle.